# Tema 4.1: SAM3 - Arquitectura Encoder-Decoder Promptable

*Duración estimada: 1 hora*

## 1. El Paradigma de Segmentación 'Promptable'

Antes de SAM, entrenar un modelo de segmentación requería miles de imágenes etiquetadas a mano para una tarea específica (ej. detectar manzanas). SAM cambió el paradigma: es un **Foundation Model** que segmenta *cualquier cosa*, siempre y cuando le des un **prompt** (una pista).

¿Qué puede ser un prompt?
- Un punto en el centro del objeto.
- Una caja delimitadora (Bounding Box) alrededor del objeto.
- Un texto libre (ej. 'encuentra el coche rojo').
- Una máscara burda generada en un frame anterior (útil para video).

## 2. Anatomía de SAM (Segment Anything Model)

SAM consta de tres piezas fundamentales:

1. **Image Encoder (Codificador de Imagen)**: Es el "cerebro visual". Normalmente es un Vision Transformer (ViT) inmenso (ViT-H, ViT-L). Analiza la imagen y crea una matriz matemática gigante llamada *Embedding*. 
   - **Regla de oro:** El Image Encoder es muy lento, pero **solo se ejecuta una vez por imagen**.

2. **Prompt Encoder (Codificador de Pista)**: Es muy ligero. Traduce tus puntos, cajas o textos en embeddings matemáticos que el modelo puede entender.

3. **Mask Decoder (Decodificador de Máscara)**: Es un transformador ultra-rápido y ligero. Toma el *Embedding de la imagen* y el *Embedding del prompt*, y los fusiona en tiempo real para generar la máscara final.

In [ ]:
import time
import numpy as np

# Vamos a simular computacionalmente esta arquitectura para entender la latencia.

class DummyImageEncoder:
    def encode(self, image):
        print("\n[Image Encoder] Procesando imagen (Esto es muy pesado)...")
        time.sleep(2.0) # Simula 2 segundos de latencia de un ViT enorme
        return np.random.rand(1, 256, 64, 64) # Representa el Image Embedding

class DummyPromptEncoder:
    def encode(self, prompt):
        print(f"[Prompt Encoder] Codificando prompt: '{prompt}' (Rápido)")
        time.sleep(0.05) # Súper rápido
        return np.random.rand(1, 256) # Representa el Prompt Embedding

class DummyMaskDecoder:
    def decode(self, image_embedding, prompt_embedding):
        print("[Mask Decoder] Fusionando embeddings y generando máscara en tiempo real...")
        time.sleep(0.05) # Súper rápido
        return np.random.rand(500, 500) > 0.5 # Retorna una máscara binaria

## 3. Demostración de Latencia Interactiva

Imagina que estás creando una aplicación web donde el usuario hace clic en una imagen para segmentarla. Si corrieras el modelo completo en cada clic, el usuario tendría que esperar 2 segundos por clic. ¡Terrible experiencia!

**La magia de SAM:** Pre-calculas el *Image Embedding* al cargar la página. Luego, en cada clic del usuario, solo ejecutas el *Prompt Encoder* y el *Mask Decoder* (0.1 segundos).

In [ ]:
# 1. Inicializamos los componentes
image_encoder = DummyImageEncoder()
prompt_encoder = DummyPromptEncoder()
mask_decoder = DummyMaskDecoder()

image = "imagen_alta_resolucion.jpg"

# 2. Paso 1: Pre-computar el embedding de la imagen (Ocurre UNA vez)
start_time = time.time()
image_embedding = image_encoder.encode(image)
print(f"-> Tiempo de Image Encoder: {time.time() - start_time:.2f} segundos")

# 3. El usuario interactúa: Hace clic en 3 lugares diferentes rápidamente
prompts_usuario = ["Punto(100, 150)", "Punto(200, 200)", "Punto(400, 50)"]

print("\n--- Usuario empieza a interactuar ---")
for prompt in prompts_usuario:
    interaction_start = time.time()
    
    # Solo ejecutamos la parte ligera de la arquitectura
    prompt_embedding = prompt_encoder.encode(prompt)
    mask = mask_decoder.decode(image_embedding, prompt_embedding)
    
    print(f"-> ¡Máscara generada en {time.time() - interaction_start:.3f} segundos! Lista para renderizar.")
    print("-"*40)

## 4. Ejercicio: Optimizando con Caching

Imagina que procesas un video a 30 FPS. ¿Por qué SAM1/SAM2 pueden ser lentos en video si calculamos el Image Encoder por cada frame?

- **Respuesta:** En video, la imagen cambia cada frame, por lo que debes volver a ejecutar el pesado `Image Encoder`.
- **Solución en SAM2.1/SAM3:** Comparten características espacio-temporales entre frames. El modelo "recuerda" el embedding del frame anterior y solo calcula la diferencia, acelerando drásticamente el proceso de codificación de imagen en flujos de video.